# Connect to the shared MCP server

Run all cells below in order. The first holds this notebook's configuration; the second resolves the current MCP host address from Bitbucket; the third prints the commands you run to connect your MCP client (Claude Code or opencode).

In [1]:
# Edit these (or re-run scripts/build_connect_notebook.py) if any of them
# ever change — everything below reads from these constants.
MAIN_HOST = 'http://127.0.0.1:8766'

# Where the current MCP host address lives in Bitbucket (Bitbucket
# Server/Data Center REST API: BITBUCKET_BASE_URL/rest/api/1.0/...).
# Change BITBUCKET_PROJECT/BITBUCKET_REPO/BITBUCKET_FILE_PATH to point at
# whatever file in whatever repo your org actually keeps this in.
BITBUCKET_BASE_URL = 'https://bitbucket.example.org'
BITBUCKET_PROJECT = 'PROJ'
BITBUCKET_REPO = 'mcp-config'
BITBUCKET_FILE_PATH = 'host.txt'


In [3]:
import getpass

import requests

bb_username = input("Bitbucket username: ")
bb_password = getpass.getpass("Bitbucket password (or personal access token): ")

raw_url = (
    f"{BITBUCKET_BASE_URL.rstrip('/')}/rest/api/1.0/projects/{BITBUCKET_PROJECT}"
    f"/repos/{BITBUCKET_REPO}/raw/{BITBUCKET_FILE_PATH}"
)
response = requests.get(raw_url, auth=(bb_username, bb_password))
response.raise_for_status()

host_address = response.text.strip()
print("Resolved MCP host:", host_address)


Bitbucket username:  123
Bitbucket password (or personal access token):  ········


ConnectionError: HTTPSConnectionPool(host='bitbucket.example.org', port=443): Max retries exceeded with url: /rest/api/1.0/projects/PROJ/repos/mcp-config/raw/host.txt (Caused by NameResolutionError("HTTPSConnection(host='bitbucket.example.org', port=443): Failed to resolve 'bitbucket.example.org' ([Errno -5] No address associated with hostname)"))

In [4]:
import getpass
import subprocess

os_username = getpass.getuser()
mcp_url = f"{host_address}/mcp"

claude_command = (
    f'claude mcp add shared-tools --transport http {mcp_url} '
    f'--header "X-MCP-User: {os_username}"'
)
opencode_command = (
    f'opencode mcp add shared-tools --url {mcp_url} '
    f'--header "X-MCP-User={os_username}"'
)

print("\nTo connect with Claude Code, run:\n")
print(claude_command)
print("\nTo connect with opencode, run:\n")
print(opencode_command)

subprocess.run(
    ["curl", "-s", "-X", "POST", 'http://127.0.0.1:8766/api/users/register', "-d", f"username={os_username}"],
    capture_output=True,
)
print("\n(You've been registered in the admin UI's user list.)")


NameError: name 'host_address' is not defined